# Stage 3 Qwen One-Pass (Kaggle)

This notebook performs the full Stage 3 run in one pass:
1. environment setup
2. preflight (1 sample)
3. full val run
4. validate + eval + visuals
5. print key metrics in notebook output
6. pack all artifacts into `.tar.gz`

Important: if you cannot return to this runtime later, download the final archive from output.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]

JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

BACKEND_MODE = "qwen_hf"
PROMPT_VERSION = "qwen_vlm_labels_v1_prompt_v3"   # switch to ..._v1 or ..._v2 for A/B
RUN_ID = f"stage3_qwen_val_v2_{PROMPT_VERSION}"

DO_PREFLIGHT = True
PREFLIGHT_SAMPLES = 1

DO_FULL_RUN = True

print("RUN_ID:", RUN_ID)


In [ ]:
def sh(cmd: str, cwd: Path | None = None, check: bool = True):
    print(f"$ {cmd}")
    p = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p

def path_exists_or_raise(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


In [ ]:
# 1) Setup: clone + deps + GPU check
if REPO_DIR.exists():
    print("Repo already exists, reusing:", REPO_DIR)
else:
    sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")

sh("python -m pip install -q -U pip")
sh(f"python -m pip install -q -r {REPO_DIR / 'requirements.txt'}")
sh("python -m pip install -q -U transformers accelerate qwen-vl-utils")

sh("nvidia-smi", check=False)
print("cwd:", REPO_DIR)


In [ ]:
# 2) Resolve dataset jsonl path
VAL_JSONL = None
for root in DATASET_ROOT_CANDIDATES:
    p = root / JSONL_REL
    if p.exists():
        VAL_JSONL = p
        break

if VAL_JSONL is None:
    found = list(Path("/kaggle/input").rglob("vlm_labels_v1_val_v2.annotated.jsonl"))
    if found:
        VAL_JSONL = found[0]

if VAL_JSONL is None:
    raise FileNotFoundError("Could not locate vlm_labels_v1_val_v2.annotated.jsonl under /kaggle/input")

print("VAL_JSONL:", VAL_JSONL)


In [ ]:
# 3) Preflight run (cheap sanity)
if DO_PREFLIGHT:
    preflight_run_id = f"{RUN_ID}_preflight"
    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "run_stage3_vlm_baseline.py"),
            "--config", str(REPO_DIR / "configs" / "pipeline" / "stage3_vlm_gt_baseline.yaml"),
            "--backend-mode", BACKEND_MODE,
            "--prompt-version", PROMPT_VERSION,
            "--input-jsonl", f'"{VAL_JSONL}"',
            "--run-id", preflight_run_id,
            "--max-samples", str(PREFLIGHT_SAMPLES),
            "--no-resume",
        ]),
        cwd=REPO_DIR,
    )
    print("Preflight done:", preflight_run_id)
else:
    print("Preflight skipped")


In [ ]:
# 4) Full run + validate + eval + visuals
if DO_FULL_RUN:
    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "run_stage3_vlm_baseline.py"),
            "--config", str(REPO_DIR / "configs" / "pipeline" / "stage3_vlm_gt_baseline.yaml"),
            "--backend-mode", BACKEND_MODE,
            "--prompt-version", PROMPT_VERSION,
            "--input-jsonl", f'"{VAL_JSONL}"',
            "--run-id", RUN_ID,
            "--no-resume",
        ]),
        cwd=REPO_DIR,
    )

    RUN_DIR = REPO_DIR / "outputs" / "stage3_vlm_baseline_runs" / RUN_ID
    PRED_JSONL = RUN_DIR / "predictions_vlm_labels_v1.jsonl"

    path_exists_or_raise(RUN_DIR, "RUN_DIR")
    path_exists_or_raise(PRED_JSONL, "predictions_vlm_labels_v1.jsonl")

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "validate_vlm_labels_v1.py"),
            "--input", f'"{PRED_JSONL}"',
        ]),
        cwd=REPO_DIR,
    )

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "eval_stage3_vlm_baseline.py"),
            "--run-dir", f'"{RUN_DIR}"',
            "--ground-truth-jsonl", f'"{VAL_JSONL}"',
        ]),
        cwd=REPO_DIR,
    )

    EVAL_DIR = RUN_DIR / "eval"
    path_exists_or_raise(EVAL_DIR / "metrics.json", "metrics.json")

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "visualize_stage3_eval_results.py"),
            "--eval-dir", f'"{EVAL_DIR}"',
        ]),
        cwd=REPO_DIR,
    )

    print("Full run done:", RUN_ID)
    print("RUN_DIR:", RUN_DIR)
else:
    raise RuntimeError("DO_FULL_RUN=False: notebook expects full run for final artifacts")


In [ ]:
# 5) Print key metrics in notebook output (so they are visible immediately)
METRICS_PATH = RUN_DIR / "eval" / "metrics.json"
RUN_SUMMARY_PATH = RUN_DIR / "run_summary.json"

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
run_summary = json.loads(RUN_SUMMARY_PATH.read_text(encoding="utf-8"))

rates = metrics.get("rates", {})
f1 = metrics.get("f1", {})
counts = metrics.get("counts", {})

summary = {
    "run_id": RUN_ID,
    "prompt_version": PROMPT_VERSION,
    "backend": run_summary.get("backend_mode_effective"),
    "model": run_summary.get("backend_settings_effective", {}).get("model"),
    "records_attempted": run_summary.get("counters", {}).get("records_attempted"),
    "status_ok": run_summary.get("counters", {}).get("status_ok"),
    "status_backend_error": run_summary.get("counters", {}).get("status_backend_error"),
    "parse_success_rate": rates.get("parse_success_rate"),
    "schema_valid_rate": rates.get("schema_valid_rate"),
    "coarse_class_accuracy": rates.get("coarse_class_accuracy"),
    "coarse_class_macro_f1": f1.get("coarse_class_macro_f1"),
    "visibility_accuracy": rates.get("visibility_accuracy"),
    "visibility_macro_f1": f1.get("visibility_macro_f1"),
    "needs_review_accuracy": rates.get("needs_review_accuracy"),
    "tag_exact_match_rate": rates.get("tag_exact_match_rate"),
    "tag_mean_jaccard": rates.get("tag_mean_jaccard"),
    "samples_evaluated": counts.get("samples_evaluated"),
}

print("=== STAGE3 RUN SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nmetrics.json:", METRICS_PATH)
print("run_summary.json:", RUN_SUMMARY_PATH)


In [ ]:
# 6) Collect deliverables + pack archive
DELIVER_DIR = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID}")
if DELIVER_DIR.exists():
    shutil.rmtree(DELIVER_DIR)
DELIVER_DIR.mkdir(parents=True, exist_ok=True)

to_copy = [
    RUN_DIR / "run_summary.json",
    RUN_DIR / "config_snapshot.json",
    RUN_DIR / "predictions_vlm_labels_v1.jsonl",
    RUN_DIR / "parsed_predictions.jsonl",
    RUN_DIR / "raw_responses.jsonl",
    RUN_DIR / "sample_results.jsonl",
    RUN_DIR / "failures.jsonl",
    RUN_DIR / "eval" / "metrics.json",
    RUN_DIR / "eval" / "confusion_coarse_class.csv",
    RUN_DIR / "eval" / "confusion_visibility.csv",
    RUN_DIR / "eval" / "review_table.csv",
    RUN_DIR / "eval" / "failures.jsonl",
    RUN_DIR / "eval" / "visuals" / "report.md",
    RUN_DIR / "eval" / "visuals" / "kpi_overview.png",
    RUN_DIR / "eval" / "visuals" / "confusion_coarse_class.png",
    RUN_DIR / "eval" / "visuals" / "confusion_visibility.png",
]

for src in to_copy:
    if src.exists():
        rel = src.relative_to(RUN_DIR)
        dst = DELIVER_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

summary_md = DELIVER_DIR / "RESULT_SUMMARY.md"
summary_md.write_text(
    "\n".join([
        f"# Stage 3 One-Pass Result: {RUN_ID}",
        "",
        f"- prompt_version: {PROMPT_VERSION}",
        f"- backend: {summary['backend']}",
        f"- model: {summary['model']}",
        f"- records_attempted: {summary['records_attempted']}",
        f"- status_ok: {summary['status_ok']}",
        f"- status_backend_error: {summary['status_backend_error']}",
        "",
        "## Main metrics",
        f"- parse_success_rate: {summary['parse_success_rate']}",
        f"- schema_valid_rate: {summary['schema_valid_rate']}",
        f"- coarse_class_accuracy: {summary['coarse_class_accuracy']}",
        f"- coarse_class_macro_f1: {summary['coarse_class_macro_f1']}",
        f"- visibility_accuracy: {summary['visibility_accuracy']}",
        f"- visibility_macro_f1: {summary['visibility_macro_f1']}",
        f"- needs_review_accuracy: {summary['needs_review_accuracy']}",
        f"- tag_exact_match_rate: {summary['tag_exact_match_rate']}",
        f"- tag_mean_jaccard: {summary['tag_mean_jaccard']}",
        "",
        f"Ground truth JSONL: {VAL_JSONL}",
        f"Run dir: {RUN_DIR}",
    ]),
    encoding="utf-8",
)

ARCHIVE_BASE = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID}")
ARCHIVE_PATH = shutil.make_archive(str(ARCHIVE_BASE), "gztar", root_dir=DELIVER_DIR)

print("DELIVER_DIR:", DELIVER_DIR)
print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("\nFiles in deliverables:")
for p in sorted(DELIVER_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(DELIVER_DIR))


## What to download after run

Download this file from `/kaggle/working/`:
- `stage3_deliverables_<RUN_ID>.tar.gz`

If you ran this as a Kaggle Kernel, you can download outputs with:
```bash
kaggle kernels output <username>/<kernel-slug> -p <local_folder>
```

The archive already contains `metrics.json`, confusion CSV, review table, predictions, run summary, and visual `report.md`.
